# Prioritize designs based on af3 design stats
All criteria are viewed in comparison to authors' successful designs. The cutoffs will be modified after receiving the nonbinding designs from the authors.

Applies all discussed filter criteria to af3_design_stats.tsv and outputs:
  1. prioritization_summary.tsv  — one row per design with flag counts and tier
  2. tier1.tsv                   — Tier 1: 0 hard flags + salt bridge or hbond at p5
  3. tier2.tsv                   — Tier 2: 0 hard flags, no salt bridge or hbond at p5
  4. redesign_candidates.tsv     — 1 hard flag (hydrophobicity only) + salt bridge/hbond at p5 + CMS score >= 25
  5. deprioritized.tsv           — 2+ hard flags or p5 CMS score < 25
  6. prioritization_report.txt   — human-readable summary
 
Hard filters (any failure = 1 hard flag):
  1. dG_separated            > -50 REU
  2. dG_separated/dSASAx100  > -2.3
  3. dSASA_int               < 2000 Å²
  4. hbonds_int / nres_int   < 0.07
  5. delta_unsatHbonds / nres_int > 0.20  (calibrated to authors: 4/18 hit >0.15,
                                         max author = 0.20 for yfv-2)
  6. surface_hydrophobicity  > 0.60       (hard; 0.35 = BindCraft soft thresh)
  7. interface_n_K           > 3
  8. ipsae_binder_peptide    < 0.80
 
Soft flags (reported but do not count toward hard flag total):
  1. sc_value                < 0.55
  2. packstat                < 0.55
  3. buns_delta_unsat        >= 4         (soft: 7/18 validated authors hit this)
  4. binder_aligned_rmsd     > 1.0        (flag for structural inspection)
 
Specificity signal (reported separately, used for tiering):
  1. n_saltbridge_hotspot_p5 >= 1         (R/K contacting G12D neoepitope)
  2. cms_hotspot_p5          (CMS at p5)
  3. n_contacts_hotspot_p5_polar
 
Tiering (after hard flags counted):
  1. Tier 1:               0 hard flags + salt bridge or hbond at p5
  2. Tier 2:               0 hard flags, no salt bridge or hbond at p5 (sorted by cms_hotspot_p5)
  3. Redesign candidate:   1 hard flag (hydrophobicity only) + salt bridge/hbond at p5 + CMS score >= 25
  4. Soft flag only:       1 hard flag (non-hydrophobicity), or soft flags only
  5. Deprioritize:         2+ hard flags
  6. Exclude:              4+ hard flags

In [6]:
import os 
import numpy as np
import pandas as pd

In [11]:
HARD_FILTERS = {
    # (column, operator, threshold, description)
    'dG_separated':           ('>', -50,   'dG_separated > -50 REU (weak binding)'),
    'dG_sep_norm':            ('>', -2.3,  'dG_separated/dSASAx100 > -2.3'),
    'dSASA_int':              ('<', 2000,  'dSASA_int < 2000 Å² (small interface)'),
    'hbonds_per_res':         ('<', 0.07,  'hbonds_int/nres_int < 0.07'),
    'delta_unsat_per_res':    ('>', 0.20,  'delta_unsatHbonds/nres_int > 0.20'),
    'surface_hydrophobicity': ('>', 0.60,  'surface_hydrophobicity > 0.60'),
    'interface_n_K':          ('>', 3,     'interface_n_K > 3'),
    'ipsae_binder_peptide':   ('<', 0.80,  'ipsae_binder_peptide < 0.80'),
    'cms_hotspot_p5':          ('<', 25.0, 'cms_hotspot_p5 < 25 (weak neoepitope contact)'),
}
 
SOFT_FLAGS = {
    'sc_value':           ('<', 0.55,  'sc_value < 0.55 (shape complementarity, soft)'),
    'packstat':           ('<', 0.55,  'packstat < 0.55 (soft)'),
    'buns_delta_unsat':   ('>=', 4,   'buns_delta_unsat >= 4 (soft; 7/18 authors)'),
    'binder_aligned_rmsd':('>', 1.0,  'binder_aligned_rmsd > 1.0 (inspect structure)'),
}
 
SALT_BRIDGE_COL  = 'n_saltbridge_hotspot_p5'
CMS_HOTSPOT_COL  = 'cms_hotspot_p5'
POLAR_CT_COL     = 'n_contacts_hotspot_p5_polar'
HBOND_CT_COL     = 'n_contacts_hotspot_p5_hbond'

In [13]:
def _check(val, op, thresh) -> bool:
    """Return True if the condition (val op thresh) is met (i.e. filter fails)."""
    if pd.isna(val):
        return False
    if op == '>':   return val > thresh
    if op == '<':   return val < thresh
    if op == '>=':  return val >= thresh
    if op == '<=':  return val <= thresh
    return False


def apply_filters(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived columns and flag columns to df.
    Returns augmented df.
    """
    df = df.copy()

    # Derived columns
    df['hbonds_per_res'] = df['hbonds_int'] / df['nres_int'].replace(0, np.nan)
    df['delta_unsat_per_res'] = df['delta_unsatHbonds'] / df['nres_int'].replace(0, np.nan)

    # Resolve dG_separated/dSASAx100 column (pandas may rename slash)
    dg_norm_col = None
    for c in df.columns:
        if c.replace('/', '').replace('x', 'X') == 'dGseparateddSASAx100'.replace('x', 'X'):
            dg_norm_col = c
            break
    if dg_norm_col and dg_norm_col != 'dG_sep_norm':
        df['dG_sep_norm'] = df[dg_norm_col]
    elif 'dG_sep_norm' not in df.columns:
        df['dG_sep_norm'] = np.nan

    # Hard flags
    hard_flag_cols = []
    for key, (op, thresh, desc) in HARD_FILTERS.items():
        col = f'flag_{key}'
        if key in df.columns:
            df[col] = df[key].apply(lambda v: _check(v, op, thresh))
        else:
            df[col] = False
        hard_flag_cols.append(col)

    df['n_hard_flags'] = df[hard_flag_cols].sum(axis=1).astype(int)

    # Which hard flags triggered
    def _flag_names(row):
        names = []
        for key, (op, thresh, desc) in HARD_FILTERS.items():
            if row.get(f'flag_{key}', False):
                names.append(key)
        return names
    df['hard_flags_triggered'] = df.apply(_flag_names, axis=1)

    # Soft flags
    soft_flag_cols = []
    for key, (op, thresh, desc) in SOFT_FLAGS.items():
        col = f'softflag_{key}'
        if key in df.columns:
            df[col] = df[key].apply(lambda v: _check(v, op, thresh))
        else:
            df[col] = False
        soft_flag_cols.append(col)
 
    df['n_soft_flags'] = df[soft_flag_cols].sum(axis=1).astype(int)

    def _soft_names(row):
        names = []
        for key in SOFT_FLAGS:
            if row.get(f'softflag_{key}', False):
                names.append(key)
        return names
    df['soft_flags_triggered'] = df.apply(_soft_names, axis=1)

    # Salt bridge and CMS
    df['has_salt_bridge_p5'] = df.get(SALT_BRIDGE_COL, pd.Series(0, index=df.index)) >= 1
    df['cms_p5']              = df.get(CMS_HOTSPOT_COL, pd.Series(np.nan, index=df.index))
    df['n_polar_contacts_p5'] = df.get(POLAR_CT_COL, pd.Series(0, index=df.index))
    df['n_hbonds_p5']         = df.get(HBOND_CT_COL, pd.Series(0, index=df.index))

    # Hydrophobicity-only hard flag (for redesign candidate check)
    df['only_hydrophob_flag'] = (
        (df['n_hard_flags'] == 1) &
        df.get('flag_surface_hydrophobicity', pd.Series(False, index=df.index))
    )

    # Tier assignment
    # Single-flag redesign conditions:
    # - hydrophobicity only + salt bridge -> redesign surface residues
    # - dSASA_int only (borderline interface size, otherwise clean) -> redesign to extend contacts
    df['only_dSASA_flag'] = (
        (df['n_hard_flags'] == 1) &
        df.get('flag_dSASA_int', pd.Series(False, index=df.index))
    )

        # Specificity signal: salt bridge OR hbond at p5
    df['has_p5_specificity'] = (
        (df.get(SALT_BRIDGE_COL, pd.Series(0, index=df.index)) >= 1) |
        (df.get(HBOND_CT_COL,    pd.Series(0, index=df.index)) >= 1)
    )

    def _tier(row):
        nf       = row['n_hard_flags']
        p5_spec  = row['has_p5_specificity']   # salt bridge OR hbond at p5
        cms_ok   = row.get('cms_hotspot_p5', 0) >= 25.0
        cms_flag = bool(row.get('flag_cms_hotspot_p5', False))
 
        # Tier 1: 0 hard flags + salt bridge or hbond at p5
        if nf == 0 and p5_spec:
            return 'Tier1'
        # Tier 2: 0 hard flags, no p5 specificity signal
        if nf == 0:
            return 'Tier2'
        # cms_hotspot_p5 flag -> always deprioritize (not correctable by redesign)
        if cms_flag:
            return 'Deprioritize'
        # Redesign candidates: 1 non-cms hard flag + p5 specificity signal + cms ok
        if nf == 1 and p5_spec and cms_ok:
            return 'Redesign_candidate'
        # 1 non-cms hard flag, no p5 specificity
        if nf == 1:
            return 'Soft_flag_only'
        if nf < 4:
            return 'Deprioritize'
        return 'Exclude'
 
    df['tier'] = df.apply(_tier, axis=1)

    return df

In [19]:
TIER_ORDER = ['Tier1', 'Tier2', 'Redesign_candidate', 'Soft_flag_only',
              'Deprioritize', 'Exclude']


def write_report(df: pd.DataFrame, out_path: str):
    lines = []
    lines.append("=" * 72)
    lines.append("DESIGN PRIORITIZATION REPORT")
    lines.append("=" * 72)
 
    lines.append("\nHARD FILTER THRESHOLDS (calibrated to author designs):")
    for key, (op, thresh, desc) in HARD_FILTERS.items():
        lines.append(f"  {desc}")
 
    lines.append("\nSOFT FLAG THRESHOLDS (informational only):")
    for key, (op, thresh, desc) in SOFT_FLAGS.items():
        lines.append(f"  {desc}")
 
    lines.append(f"\nTotal designs: {len(df)}")
 
    for tier in TIER_ORDER:
        sub = df[df['tier'] == tier].sort_values('cms_p5', ascending=False)
        lines.append(f"\n{'─'*72}")
        lines.append(f"{tier.upper()}  (n={len(sub)})")
        lines.append(f"{'─'*72}")
        for _, r in sub.iterrows():
            sb_str  = f"  SB={int(r.get(SALT_BRIDGE_COL, 0))}" if r['has_salt_bridge_p5'] else ''
            cms_str = f"  cms_p5={r['cms_p5']:.1f}" if pd.notna(r['cms_p5']) else ''
            pc_str  = f"  polar_ct_p5={int(r.get(POLAR_CT_COL, 0))}"
            hb_str  = f"  hbonds_p5={int(r.get(HBOND_CT_COL, 0))}"
            ipSAE   = f"  ipSAE={r['ipsae_binder_peptide']:.3f}" if 'ipsae_binder_peptide' in r and pd.notna(r['ipsae_binder_peptide']) else ''
            hard_str = f"  hard_flags={r['hard_flags_triggered']}" if r['hard_flags_triggered'] else ''
            soft_str = f"  soft_flags={r['soft_flags_triggered']}" if r['soft_flags_triggered'] else ''
            lines.append(f"  {r['design']:<40}{sb_str}{cms_str}{pc_str}{hb_str}{ipSAE}{hard_str}{soft_str}")
 
    lines.append(f"\n{'='*72}")
    lines.append("FILTER FAILURE COUNTS (hard filters)")
    lines.append(f"{'='*72}")
    for key, (op, thresh, desc) in HARD_FILTERS.items():
        col = f'flag_{key}'
        if col in df.columns:
            n = df[col].sum()
            lines.append(f"  {desc}: {n}/{len(df)} fail")
 
    lines.append(f"\n{'='*72}")
    lines.append("SALT BRIDGE SUMMARY (n_saltbridge_hotspot_p5 >= 1)")
    lines.append(f"{'='*72}")
    sb_df = df[df['has_salt_bridge_p5']].sort_values('cms_p5', ascending=False)
    for _, r in sb_df.iterrows():
        lines.append(
            f"  {r['design']:<40}  cms_p5={r['cms_p5']:.1f}"
            f"  polar_ct={int(r.get(POLAR_CT_COL,0))}"
            f"  hbonds_p5={int(r.get(HBOND_CT_COL,0))}"
            f"  SB={int(r.get(SALT_BRIDGE_COL,0))}"
            f"  sb_resnums={r.get('saltbridge_hotspot_p5_binder_resnums','')}"
            f"  sb_aas={r.get('saltbridge_hotspot_p5_binder_aas','')}"
            f"  tier={r['tier']}"
            f"  hard_flags={r['hard_flags_triggered']}"
        )
 
    with open(out_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print(f"Report -> {out_path}")

In [18]:
def main(stats_tsv, out_dir, hbond_tsv):
    '''
    input:
    stats_tsv  af3_design_stats.tsv (with merged FastRelax columns)
    hbond_tsv hbond_contacts.tsv (from af3_design_stats.py)
    out_dir
    ''' 
    os.makedirs(out_dir, exist_ok=True)
 
    df_raw = pd.read_csv(stats_tsv, sep='\t')
    print(f"Loaded {len(df_raw)} designs from {stats_tsv}")
 
    # Build hbond hotspot p5 residue lookup from hbond_contacts.tsv
    hbond_p5_resnums = {}
    hbond_p5_aas     = {}
    if hbond_tsv and os.path.exists(hbond_tsv):
        df_hb = pd.read_csv(hbond_tsv, sep='\t')
        # Filter to hotspot p5 contacts (pep_pos_1idx == 5 and is_hotspot == True)
        # Use is_hotspot if available, else filter by pep_pos_1idx matching hotspot
        if 'is_hotspot' in df_hb.columns:
            hb_p5 = df_hb[df_hb['is_hotspot'] == True]
        else:
            hb_p5 = df_hb[df_hb['pep_pos_1idx'] == 5]
        for design, grp in hb_p5.groupby('design'):
            resnums = sorted(grp['binder_resnum'].dropna().astype(int).unique().tolist())
            aas     = grp['binder_aa'].dropna().tolist()
            hbond_p5_resnums[design] = str(resnums)
            hbond_p5_aas[design]     = str(aas)
        print(f"Loaded hbond contacts for {len(hbond_p5_resnums)} designs from {hbond_tsv}")
    elif hbond_tsv:
        print(f"WARNING: --hbond_tsv not found: {hbond_tsv}")
 
    df_raw['hbond_hotspot_p5_binder_resnums'] = df_raw['design'].map(hbond_p5_resnums).fillna('[]')
    df_raw['hbond_hotspot_p5_binder_aas']     = df_raw['design'].map(hbond_p5_aas).fillna('[]')
 
    df = apply_filters(df_raw)
 
    # Summary columns for output TSVs
    summary_cols = [
        'design',
        'tier', 'n_hard_flags', 'hard_flags_triggered',
        'n_soft_flags', 'soft_flags_triggered',
        'has_salt_bridge_p5', SALT_BRIDGE_COL, CMS_HOTSPOT_COL, POLAR_CT_COL, HBOND_CT_COL,
        'flag_cms_hotspot_p5',
        'ipsae_binder_peptide', 'ipsae_n_contacts',
        'dG_separated', 'dG_sep_norm', 'dSASA_int',
        'hbonds_int', 'hbonds_per_res',
        'delta_unsatHbonds', 'delta_unsat_per_res',
        'sc_value', 'packstat',
        'surface_hydrophobicity', 'buns_delta_unsat',
        'interface_n_K', 'interface_n_M',
        'binder_aligned_rmsd', 'target_aligned_rmsd',
        'n_binder_res', 'ss_helix_pct', 'ss_loop_pct',
        'net_charge', 'pass_all',
        'saltbridge_hotspot_p5_binder_resnums', 'saltbridge_hotspot_p5_binder_aas',
        'hbond_hotspot_p5_binder_resnums', 'hbond_hotspot_p5_binder_aas',
    ]
    present = [c for c in summary_cols if c in df.columns]
 
    # Full summary TSV
    df_out = df[present].copy()
    df_out['hard_flags_triggered'] = df_out['hard_flags_triggered'].astype(str)
    df_out['soft_flags_triggered'] = df_out['soft_flags_triggered'].astype(str)
    df_out = df_out.sort_values(
        ['n_hard_flags', 'has_salt_bridge_p5', CMS_HOTSPOT_COL],
        ascending=[True, False, False]
    )
    summary_path = os.path.join(out_dir, 'prioritization_summary.tsv')
    df_out.to_csv(summary_path, sep='\t', index=False)
    print(f"Summary -> {summary_path}")
 
    # Per-tier TSVs
    HBOND_COLS = ['hbond_hotspot_p5_binder_resnums', 'hbond_hotspot_p5_binder_aas']
    HBOND_TIERS = {'Tier1', 'Tier2', 'Redesign_candidate'}
    for tier in TIER_ORDER:
        cols_for_tier = present if tier in HBOND_TIERS else [c for c in present if c not in HBOND_COLS]
        sub = df[df['tier'] == tier][cols_for_tier].copy()
        if 'hard_flags_triggered' in sub.columns:
            sub['hard_flags_triggered'] = sub['hard_flags_triggered'].astype(str)
        if 'soft_flags_triggered' in sub.columns:
            sub['soft_flags_triggered'] = sub['soft_flags_triggered'].astype(str)
        sub = sub.sort_values(CMS_HOTSPOT_COL, ascending=False)
        fname = tier.lower() + '.tsv'
        sub.to_csv(os.path.join(out_dir, fname), sep='\t', index=False)
        print(f"  {tier}: n={len(sub)} -> {fname}")
 
    # Text report
    write_report(df, os.path.join(out_dir, 'prioritization_report.txt'))
 
    # Console summary
    print(f"\n── Tier summary ──")
    counts = df['tier'].value_counts().reindex(TIER_ORDER, fill_value=0)
    for tier, n in counts.items():
        print(f"  {tier:<25} {n}")

#     print(f"\n── Salt bridge designs ──")
#     sb = df[df['has_salt_bridge_p5']].sort_values('cms_p5', ascending=False)
#     for _, r in sb.iterrows():
#         print(f"  {r['design']:<45} tier={r['tier']:<22} "
#               f"cms_p5={r['cms_p5']:.1f}  polar_ct={int(r.get(POLAR_CT_COL,0))}"
#               f"  hbonds_p5={int(r.get(HBOND_CT_COL,0))}"
#               f"  SB={int(r.get(SALT_BRIDGE_COL,0))}"
#               f"  resnums={r.get('saltbridge_hotspot_p5_binder_resnums','')}"
#               f"  aas={r.get('saltbridge_hotspot_p5_binder_aas','')}"
#               f"  hard_flags={r['hard_flags_triggered']}")

In [20]:
main('/n/groups/marks/users/aaron/pmhc_cp/post_filter/outputs/r2/af3_nomsa/af3_design_stats.tsv', 
     '/n/groups/marks/users/aaron/pmhc_cp/post_filter/outputs/r2/prioritization/',
     '/n/groups/marks/users/aaron/pmhc_cp/post_filter/outputs/r2/af3_nomsa/contacts/hbond_contacts.tsv'
    )

Loaded 49 designs from /n/groups/marks/users/aaron/pmhc_cp/post_filter/outputs/r2/af3_nomsa/af3_design_stats.tsv
Loaded hbond contacts for 12 designs from /n/groups/marks/users/aaron/pmhc_cp/post_filter/outputs/r2/af3_nomsa/contacts/hbond_contacts.tsv
Summary -> /n/groups/marks/users/aaron/pmhc_cp/post_filter/outputs/r2/prioritization/prioritization_summary.tsv
  Tier1: n=11 -> tier1.tsv
  Tier2: n=2 -> tier2.tsv
  Redesign_candidate: n=4 -> redesign_candidate.tsv
  Soft_flag_only: n=1 -> soft_flag_only.tsv
  Deprioritize: n=31 -> deprioritize.tsv
  Exclude: n=0 -> exclude.tsv
Report -> /n/groups/marks/users/aaron/pmhc_cp/post_filter/outputs/r2/prioritization/prioritization_report.txt

── Tier summary ──
  Tier1                     11
  Tier2                     2
  Redesign_candidate        4
  Soft_flag_only            1
  Deprioritize              31
  Exclude                   0


/tmp/ipykernel_780099/772440640.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['hbond_hotspot_p5_binder_resnums'] = df_raw['design'].map(hbond_p5_resnums).fillna('[]')
/tmp/ipykernel_780099/772440640.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['hbond_hotspot_p5_binder_aas']     = df_raw['design'].map(hbond_p5_aas).fillna('[]')
